# [DIST PRE-PHASE] DQN→PPO Distillation — Run All Notebook

**TEMPORARY notebook — remove at graduation.**

This notebook is designed for **Runtime → Run all**. You do not need to paste anything between cells.

---

## Full lifecycle, in plain English

**A frozen pre-trained DQN teaches PPO one thing — direction (BUY vs SELL) — then retires itself once PPO proves it has learned the strategy.**

### Stage 1 — Setup (you are here)
- Click **Runtime → Run all**.
- Cells clone the branch, download the DQN checkpoint if missing, probe its dims, build the env, run a sanity check, and launch training.
- Kill switch: set `ENABLE_DIST = False` in Cell 2 to run the base repo with no distillation.

### Stage 2 — DIST_PRE_PHASE (DQN teaching)
- Wide drawdown (5%), low daily target (2%), distillation weight 0.30.
- DQN actively biases PPO toward agreement on entry direction.
- Transitions automatically to **DIST_PHASE_1** at the first day-end.
- **You do nothing here.**

### Stage 3 — DIST_PHASE_1 (DQN fading, full FTMO rules)
- Real FTMO rules resume: 1% max daily DD, 2.5% daily target.
- The DQN's influence fades automatically based on PPO's gate progress.
- **Graduation gate — ALL THREE must pass:**
  1. **Performance** — 10 consecutive days where `win_rate > 0.55`, `profit_factor > 1.3`, `expectancy_pips > 0`, `≥ 3 trades`, and no DD breach. A single bad day resets the streak.
  2. **Convergence** — 5-day rolling normalized agreement ≥ 0.30 (raw ≥65%; baseline is 50% because FLAT is masked on entry).
  3. **Independence** — 3-day solo dry run with DQN silent, all 3 days passing Signal 1 criteria. Failure resets the Signal 1 streak to 0.
- **You do nothing here either.** The manager runs the gate, fades the weight, and triggers the solo run automatically.
- **How to know PPO has learned the strategy:** wait for the giant banner in your training logs:
  ```
  [DIST] GRADUATION PROOF COMPLETE ✅
  Signal 1: 10/10 consecutive gate days passed
  Signal 2: XX.X% agreement (normalized: 0.YY)
  Signal 3: Solo dry run — 3/3 days passed
  → DQN RETIRING. Graduation record written.
  ```

### Stage 4 — Graduation and removal
- The dist system writes `dist_graduation_record.json` to Drive. **Save this file permanently** — it is your audit trail.
- PPO continues training fully autonomously; DQN is no longer called.
- Within a few weeks, remove the dist code:
  1. Delete the bookended blocks in `core/settings.py` and `core/pipeline.py`.
  2. `rm -rf core/dist_teacher/ core/dist_phase/ tests/dist/`
  3. `rm scripts/dist_checkpoint_probe.py docs/dist_prephase_colab_cells.md dist_prephase_run_all.ipynb`
  4. `pytest tests/` — every existing test must still pass.
  5. `grep -r 'DIST PRE-PHASE' .` must return empty.
  6. First training run after removal must use `--force-fresh` (the actor head was built with 3 extra slots during DIST stages).

### What to do if something goes wrong
| Banner you see | What it means | What to do |
|---|---|---|
| `[DIST] DIST_PRE_PHASE STARTED` | Stage 2 just began | Wait. |
| `[DIST] DIST_PHASE_1 STARTED` | Stage 3 just began | Wait, watch streak climb. |
| `[DIST] SOLO DRY RUN BEGIN` | 3-day independence test | Wait 3 days. |
| `[DIST] SOLO DRY RUN FAILED` | PPO not ready to solo | Don't panic. Streak resets; cooldown 3 days; manager re-tries. |
| `[DIST] GRADUATION PROOF COMPLETE` | PPO learned the strategy | Stage 4. |

The single source of truth for all of this is the header comment at the top of `core/dist_phase/dist_phase_manager.py`. If you (or an LLM) ever forget what to do, read that file's header.

---

## Cell 1 — Full environment setup (Run All safe)
Mounts Drive, clones the dist branch, and installs Python packages. TA-Lib is installed by `pip` from the prebuilt manylinux wheel (`TA-Lib>=0.6.7` in requirements.txt) — no C-library build needed.

In [ ]:
# [DIST PRE-PHASE START — REMOVE AT GRADUATION]
import os, sys, subprocess

# ── 1a. Mount Google Drive (Colab only) ─────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')

# ── 1b. Clone the dist branch ───────────────────────────────────────────────
REPO_DIR = '/content/rl-trading-live' if IN_COLAB else os.path.expanduser('~/rl-trading-live')
BRANCH   = 'feature/multi-tf-obs'
REPO_URL = 'https://github.com/monty313/rl-trading-live.git'

if IN_COLAB and os.path.exists(REPO_DIR):
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── 1c. Install Python requirements ─────────────────────────────────────────
# TA-Lib>=0.6.7 ships prebuilt manylinux wheels — no apt or source build needed.
# numpy<3.0, faiss-cpu>=1.9, torch>=2.2 are all resolved against Colab's stack.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
    check=True,
)

# ── 1d. Smoke-test imports ───────────────────────────────────────────────────
import torch, talib as _talib
print(f'[DIST OK] torch  : {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'[DIST OK] TA-Lib : {_talib.__version__}')
print(f'[DIST OK] repo   : {REPO_DIR} (branch {BRANCH})')
# [DIST PRE-PHASE END]

## Cell 2 — Configuration switches & DQN checkpoint

**All on/off switches live here.** Set `ENABLE_DIST = False` to skip distillation entirely.
Set `ENABLE_MULTI_TF = False` to fall back to single-timeframe obs (lower VRAM).
Looks at `CHECKPOINT_PATH` first; if missing, downloads from Drive.

In [ ]:
# [DIST PRE-PHASE START — REMOVE AT GRADUATION]

# ── Feature switches ─────────────────────────────────────────────────────────
ENABLE_DIST      = True   # Teacher (DQN) hints ON  — set False to skip distillation
ENABLE_MULTI_TF  = True   # Multi-timeframe obs (1m/15m/1h/1d) — set False to save VRAM
WARM_START_DQN   = True   # PPO inherits DQN input projection at init

# ── DQN checkpoint ───────────────────────────────────────────────────────────
CHECKPOINT_PATH = '/content/drive/MyDrive/checkpoints/eurusd_gpu_ph0_ep0120.pt'
GDRIVE_FILE_ID  = '1s1sC0OFBnbEicgEnkhAzHcw4qiJt1Kvc'

# Drive is already mounted in Cell 1 (section 1d); safe to use /content/drive here
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
if not os.path.exists(CHECKPOINT_PATH):
    print('[DIST] Checkpoint missing locally — downloading from Drive...')
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        import gdown
    gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}', CHECKPOINT_PATH, quiet=False)

assert os.path.exists(CHECKPOINT_PATH), f'[DIST STOP] Checkpoint not at {CHECKPOINT_PATH}'
size_gb = os.path.getsize(CHECKPOINT_PATH) / 1e9
print(f'[DIST OK] Checkpoint present: {CHECKPOINT_PATH} ({size_gb:.2f} GB)')
print(f'[DIST OK] ENABLE_DIST={ENABLE_DIST} | ENABLE_MULTI_TF={ENABLE_MULTI_TF} | WARM_START_DQN={WARM_START_DQN}')
# [DIST PRE-PHASE END]

## Cell 3 — Run the checkpoint probe


In [ ]:
# [DIST PRE-PHASE START — REMOVE AT GRADUATION]
from core.dist_teacher.dist_dqn_teacher import _find_state_dict, _infer_input_dim, _infer_output_dim
import torch

_ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
_sd = _find_state_dict(_ckpt)
assert _sd is not None, '[DIST STOP] Could not locate policy state_dict in checkpoint'
DQN_INPUT_DIM,  _in_key  = _infer_input_dim(_sd)
DQN_OUTPUT_DIM, _out_key = _infer_output_dim(_sd)
print(f'[DIST OK] DQN policy input  dim: {DQN_INPUT_DIM}  (from {_in_key})')
print(f'[DIST OK] DQN policy output dim: {DQN_OUTPUT_DIM} (from {_out_key})')
if DQN_OUTPUT_DIM != 3:
    print(f'[DIST WARN] DQN output dim is {DQN_OUTPUT_DIM}, not 3 — verify action_order in config.')
del _ckpt, _sd
# [DIST PRE-PHASE END]

## Cell 4 — Build the pipeline


In [ ]:
# [DIST PRE-PHASE START — REMOVE AT GRADUATION]
from core.settings import CFG, get_device, auto_tune_batch
from core.pipeline import build_pipeline

device = get_device()
cfg = auto_tune_batch(dict(CFG), device)
cfg['dist_prephase_enabled'] = bool(ENABLE_DIST)
cfg['MULTI_TF_OBS']          = bool(ENABLE_MULTI_TF)
cfg['WARM_START_FROM_DQN']   = bool(WARM_START_DQN)
cfg['dist_teacher']['checkpoint_path'] = CHECKPOINT_PATH
cfg.setdefault('DATA_CSV_EURUSD', None)

# Cell 4 is a sanity check only. The real training run is Cell 6, which
# uses training/train.py and loads the YAML phase dict properly.
# build_pipeline's `phase=` MUST be Optional[dict] (None or a dict from
# config/phases.yaml), never a string.
try:
    import yaml
    with open('config/phases.yaml') as _f:
        _data = yaml.safe_load(_f) or {}
    _phases = sorted(_data.get('phases', []), key=lambda p: p.get('order', 0))
    _phase  = _phases[0] if _phases else None
    print(f'[DIST OK] Loaded phase: {_phase.get("name") if _phase else None}')
except Exception as _e:
    print(f'[DIST WARN] Could not load phases.yaml ({_e}); using CFG defaults')
    _phase = None
env, agent, sizer, guard, gate = build_pipeline(cfg, device, phase=_phase)
print(f'[DIST OK] Pipeline built | env.state_dim={env.state_dim} | device={device}')
# [DIST PRE-PHASE END]

## Cell 5 — 1-step sanity check


In [ ]:
# [DIST PRE-PHASE START — REMOVE AT GRADUATION]
import torch

state = env.reset()
print(f'reset state shape: {tuple(state.shape)}')
B = getattr(env, 'B', 1)
actions = {
    'direction': torch.randint(0, 3, (B,), device=device).long(),
    'lot_raw':   torch.rand(B, device=device),
    'exit':      torch.zeros(B, device=device, dtype=torch.long),
}
next_state, reward, done, info = env.step(actions)
if isinstance(info, dict) and 'dist_bonus' in info:
    print(f'dist_weight: {info["dist_weight"]:.3f}')
    print(f'dqn_active:  {info["dqn_active"]}')
    print('[DIST OK] Sanity check passed — wrapper is firing correctly')
else:
    print('[DIST INFO] dist disabled — env returning base info only')
# [DIST PRE-PHASE END]

## Cell 6 — Launch training
**Edit `DATA_CSV` if your EURUSD features file is in a different Drive location.**

In [ ]:
# [DIST PRE-PHASE START — REMOVE AT GRADUATION]
import datetime, collections

RUN_NAME       = f'dist_prephase_{datetime.datetime.now():%Y%m%d_%H%M}'
CHECKPOINT_DIR = f'/content/drive/MyDrive/checkpoints/{RUN_NAME}'
METRICS_DIR    = f'/content/drive/MyDrive/metrics/{RUN_NAME}'
MANIFEST       = f'/content/drive/MyDrive/checkpoints/{RUN_NAME}/manifest.json'
# Real path on your Drive. If you renamed/moved your CSV, edit this.
DATA_CSV       = '/content/drive/MyDrive/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# Fail loud BEFORE launching the subprocess if the data file is missing.
if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(
        f'CSV not found: {DATA_CSV}\n'
        f'  - Is Drive mounted? (Cell 1 should have done this)\n'
        f'  - Try: ls /content/drive/MyDrive/RL-Trading-Data/\n'
        f'  - If your file is elsewhere, edit DATA_CSV above.'
    )

env_overrides = os.environ.copy()
env_overrides['DIST_PREPHASE_ENABLED'] = '1' if ENABLE_DIST else '0'
env_overrides['MULTI_TF_OBS']          = '1' if ENABLE_MULTI_TF else '0'
env_overrides['WARM_START_FROM_DQN']   = '1' if WARM_START_DQN else '0'
env_overrides['PYTHONUNBUFFERED']      = '1'  # force-flush so Colab streams in real time

cmd = [
    sys.executable, '-u', 'training/train.py',
    '--csv', DATA_CSV,
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--metrics-dir',    METRICS_DIR,
    '--manifest',       MANIFEST,
    '--start-phase', '0',
]
print('Launching:', ' '.join(cmd))
print('Logs stream below. Runtime → Interrupt to stop.')
print('═' * 70)

# Stream stdout + stderr live AND keep a rolling tail so we can dump it on failure.
proc = subprocess.Popen(
    cmd, env=env_overrides,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
tail = collections.deque(maxlen=400)   # last ~400 lines for post-mortem
try:
    for line in iter(proc.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
        tail.append(line)
    proc.stdout.close()
    rc = proc.wait()
except KeyboardInterrupt:
    proc.terminate(); proc.wait()
    raise

print('═' * 70)
print(f'training/train.py exited with code {rc}')
if rc != 0:
    print('\n────── FAILURE — last 50 lines of output ──────')
    for line in list(tail)[-50:]:
        sys.stdout.write(line)
    raise RuntimeError(f'train.py failed (exit {rc}). See full log above; tail re-dumped under FAILURE banner.')
# [DIST PRE-PHASE END]
